# Fine-tune BART for English Summarization (Kaggle)

This notebook fine-tunes [`facebook/bart-base`](https://huggingface.co/facebook/bart-base) on a sample of [CNN/DailyMail](https://huggingface.co/datasets/cnn_dailymail) to produce abstractive summaries of English news articles.

### How to run on Kaggle
1. Create a new Notebook at [kaggle.com](https://www.kaggle.com).
2. **Settings → Accelerator → GPU T4 x2** (any GPU works; CPU is slow).
3. **Settings → Internet → On** (needed to download models/data).
4. Paste these cells and run them top-to-bottom.

When it finishes, the weights are saved under `./bart-finetuned-cnn`. Download that folder from the Kaggle **Output** tab and place it in your repo at `models/english/` so the Streamlit app auto-detects and uses it.

In [ ]:
# Install training dependencies (torch with GPU support is preinstalled on Kaggle).
!pip install -q transformers datasets evaluate rouge_score nltk accelerate sentencepiece


In [ ]:
import nltk
import numpy as np
import torch

import evaluate
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

nltk.download("punkt")

print("CUDA available:", torch.cuda.is_available())

MODEL_ID = "facebook/bart-base"
DATASET = "abisee/cnn_dailymail"
DATASET_VERSION = "3.0.0"

# Use a small subset for a fast demo run. Delete the .select() calls to train on full data.
TRAIN_SIZE = 2000
EVAL_SIZE = 200

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128


In [ ]:
dataset = load_dataset(DATASET, DATASET_VERSION)

train = dataset["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
valid = dataset["validation"].shuffle(seed=42).select(range(EVAL_SIZE))
test = dataset["test"].shuffle(seed=42).select(range(EVAL_SIZE))

print("Article :", train[0]["article"][:200])
print("---")
print("Summary :", train[0]["highlights"][:200])


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)


In [ ]:
def preprocess_function(examples):
    inputs = examples["article"]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(examples["highlights"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train.map(preprocess_function, batched=True, remove_columns=train.column_names)
tokenized_valid = valid.map(preprocess_function, batched=True, remove_columns=valid.column_names)


In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

rouge = evaluate.load("rouge")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    pad = int(tokenizer.pad_token_id)

    # Drop any negative/invalid sentinel id (e.g. -100) before decoding, so the
    # tokenizer never sees an out-of-range value on any transformers version.
    def clean_ids(seqs):
        if hasattr(seqs, "tolist"):
            seqs = seqs.tolist()
        out = []
        for seq in seqs:
            cleaned = []
            for t in seq:
                try:
                    v = int(t)
                except Exception:
                    continue
                if v < 0:
                    continue
                cleaned.append(v)
            out.append(cleaned)
        return out

    preds = clean_ids(predictions)
    labs = clean_ids(labels)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labs, skip_special_tokens=True)

    decoded_preds = ["\n".join(nltk.sent_tokenize(p.strip())) for p in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(l.strip())) for l in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: round(v * 100, 2) for k, v in result.items()}
    result["gen_len"] = round(float(np.mean([sum(1 for t in p if t != pad) for p in preds])), 2)
    return result


In [ ]:
output_dir = "./bart-finetuned-cnn"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    logging_steps=50,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()


In [ ]:
# Evaluate on the validation subset.
metrics = trainer.evaluate(eval_dataset=tokenized_valid, max_length=MAX_TARGET_LENGTH, num_beams=4)
print(metrics)

# Save weights for the Streamlit app.
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Saved to", output_dir)


In [ ]:
# Try it on a held-out example.
article = test[0]["article"]
inputs = tokenizer(article, max_length=MAX_INPUT_LENGTH, truncation=True, return_tensors="pt").to(model.device)
summary_ids = model.generate(**inputs, max_length=128, min_length=30, num_beams=4, early_stopping=True)
print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))


### (Optional) Push to Hugging Face Hub
```python
from huggingface_hub import notebook_login
notebook_login()
model.push_to_hub("your-username/bart-finetuned-cnn")
tokenizer.push_to_hub("your-username/bart-finetuned-cnn")
```

### Use it in the app
1. Download `bart-finetuned-cnn` from Kaggle → **Output** tab.
2. Put the folder at `models/english/` in the repo.
3. The app auto-detects `models/english/` and uses it instead of `facebook/bart-large-cnn`.